# LightGBM to XGBoost Complete Flow - watsonx.ai Deployment

This notebook provides a complete end-to-end pipeline for:
1. Training LightGBM LambdaMART model
2. Converting LightGBM model to XGBoost format
3. Verifying model conversion accuracy
4. Deploying to watsonx.ai deployment space
5. Testing the deployed model

## Model Information
- **Original Algorithm**: LightGBM LambdaMART
- **Target Algorithm**: XGBoost LambdaMART  
- **Objective**: rank:ndcg
- **Deployment Type**: xgboost_2.0
- **Python Version**: 3.11

## IBM watsonx.ai Python SDK Documentation
**https://ibm.github.io/watsonx-ai-python-sdk/**

## Table of Contents

1. Setup and Configuration
2. Load Training Data
   - 2.1. Generate Complete Training Data with All 9 Product Categories
3. Feature Engineering
4. Train LightGBM Model
5. Convert to XGBoost
6. Verify Conversion
7. Save XGBoost Model
8. Test Model Locally
9. Connect to watsonx.ai
   - 9.1. Configuration
     - 9.1.1. Environment: CP4D (on-prem)
     - 9.1.2. Environment: IBM Cloud
   - 9.2. Set Default Target Project & Space
10. Store Model in watsonx.ai
11. Deploy Model
12. Test Deployed Model
13. Summary

## 1. Setup and Configuration

Required packages:
  - lightgbm>=4.2.0,<=4.3.0
  - numpy==1.26.4
  - pandas==2.1.4
  - xgboost==2.0.3
  - scikit-learn==1.3.0
  - ibm-watsonx-ai==1.5.3
  - ibm-cos-sdk==2.14.2

**These packages are pre-installed by default in the Python 3.11 Runtime 24.1.**

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
import pickle
import tarfile
import json
import os
from datetime import datetime, timezone, timedelta
from importlib.metadata import version
import warnings
warnings.filterwarnings('ignore')

# Set timezone to UTC+8 (Asia/Taipei)
TZ_UTC8 = timezone(timedelta(hours=8))

# Get package versions
lgb_version = version('lightgbm')
xgb_version = version('xgboost')
np_version = version('numpy')
pd_version = version('pandas')
sklearn_version = version('scikit-learn')
watsonx_version = version('ibm-watsonx-ai')
cos_version = version('ibm-cos-sdk')

print("="*80)
print("PACKAGE VERSIONS")
print("="*80)
print(f"✅ LightGBM version: {lgb_version}")
print(f"✅ XGBoost version: {xgb_version}")
print(f"✅ NumPy version: {np_version}")
print(f"✅ Pandas version: {pd_version}")
print(f"✅ Scikit-learn version: {sklearn_version}")
print(f"✅ IBM watsonx.ai version: {watsonx_version}")
print(f"✅ IBM COS SDK version: {cos_version}")
print("\n✅ All libraries imported successfully")

PACKAGE VERSIONS
✅ LightGBM version: 4.2.0
✅ XGBoost version: 2.0.3
✅ NumPy version: 1.26.4
✅ Pandas version: 2.1.4
✅ Scikit-learn version: 1.3.0
✅ IBM watsonx.ai version: 1.5.3
✅ IBM COS SDK version: 2.14.2

✅ All libraries imported successfully


In [2]:
# ============================================================================
# Helper Function: list_files_formatted()
# 輔助函數：格式化顯示本地檔案列表
# ============================================================================
# This function provides a formatted view of local files with:
# - File type icons (📦 .pkl, 📋 .json, 🗜️ .tar.gz, 🐍 .py, 📝 .md)
# - Human-readable file sizes (auto-converts to KB/MB/GB)
# - Formatted modification timestamps
# - Total file count
# ============================================================================

def list_files_formatted():
    """
    Display local files with formatted output
    顯示本地檔案的格式化輸出
    """
    import os
    from datetime import datetime
    
    def format_size(bytes):
        """Format file size"""
        for unit in ['B', 'KB', 'MB', 'GB']:
            if bytes < 1024.0:
                return f"{bytes:7.1f} {unit}"
            bytes /= 1024.0
        return f"{bytes:7.1f} TB"
    
    print("=" * 110)
    print("📁 LOCAL FILES")
    print("=" * 110)
    print()
    
    # Get all files
    files = []
    for item in os.listdir('.'):
        if os.path.isfile(item):
            stat = os.stat(item)
            files.append({
                'name': item,
                'size': stat.st_size,
                'mtime': stat.st_mtime
            })
    
    # Sort by modification time (newest first)
    files.sort(key=lambda x: x['mtime'], reverse=True)
    
    # Print header
    print(f"{'File Name':<65} {'Size':>12} {'Modified':>20}")
    print("-" * 110)
    
    # Print files with icons
    for f in files:
        name = f['name']
        size = format_size(f['size'])
        mtime = datetime.fromtimestamp(f['mtime']).strftime('%Y-%m-%d %H:%M:%S')
        
        # Add icon based on file type
        if name.endswith('.pkl'):
            icon = '📦'
        elif name.endswith('.json'):
            icon = '📋'
        elif name.endswith(('.tar.gz', '.zip')):
            icon = '🗜️'
        elif name.endswith('.py'):
            icon = '🐍'
        elif name.endswith('.md'):
            icon = '📝'
        else:
            icon = '📄'
        
        print(f"{icon} {name:<62} {size:>12} {mtime:>20}")
    
    print()
    print(f"✅ Total: {len(files)} files")
    print("=" * 110)

print("✅ list_files_formatted() function defined")

✅ list_files_formatted() function defined


## 2. Load Training Data

### Utilize the "Code Snippets" feature on Jupyter Notebook Editor to generate a code snippet to load data from a data asset or connection into your notebook!

In [ ]:
# Insert the code snippet here!!! ⤵️
#
# import os, types
# import pandas as pd
# from botocore.client import Config
# import ibm_boto3
# 
# ...
# 
# 
# 

#### Example:

In [ ]:
# # Load data from IBM Cloud Object Storage
# import os, types
# import pandas as pd
# from botocore.client import Config
# import ibm_boto3

# def __iter__(self): return 0

# # @hidden_cell
# # The following code accesses a file in your IBM Cloud Object Storage. It includes your credentials.
# # You might want to remove those credentials before you share the notebook.

# cos_client = ibm_boto3.client(service_name='s3',
#     ibm_api_key_id='auto-generated_ibm_api_key_id',
#     ibm_auth_endpoint="https://iam.cloud.ibm.com/identity/token",
#     config=Config(signature_version='oauth'),
#     endpoint_url='https://s3.direct.au-syd.cloud-object-storage.appdomain.cloud')

# bucket = 'bucket_name'
# object_key = 'LTR_model_input.csv'

# body = cos_client.get_object(Bucket=bucket,Key=object_key)['Body']
# # add missing __iter__ method, so pandas accepts body as file-like object
# if not hasattr(body, "__iter__"): body.__iter__ = types.MethodType( __iter__, body )

# df_input = pd.read_csv(body)

# print(f"✅ Input data loaded: {df_input.shape}")
# print("\nFirst few rows:")
# display(df_input.head())

✅ Input data loaded: (5, 5)

First few rows:


,cust_id,product_id(has 9 different categories),share_feature_1,product_feature_2,"relevance_score(1,2,3)\n3 > successfully applied prodcut in recent month\n2 > in the applied process but not successfully applied in recent month\n1 > only browse prodcut info via bank app in recent month"
0,a01,pl,100,32,1
1,a01,hy,100,21,3
2,b01,pl,500,0,3
3,b01,hy,500,74,1
4,b01,ins,500,32,1


### Clean column names

In [4]:
# Clean column names
df_input.columns = df_input.columns.str.strip()
column_mapping = {
    'cust_id': 'cust_id',
    'product_id(has 9 different categories)': 'product_id',
    'share_feature_1': 'share_feature_1',
    'product_feature_2': 'product_feature_2'
}
relevance_col = [col for col in df_input.columns if 'relevance_score' in col][0]
column_mapping[relevance_col] = 'relevance_score'
df = df_input.rename(columns=column_mapping)
print(f"✅ Columns cleaned")
print(f"Data shape: {df.shape}")

✅ Columns cleaned
Data shape: (5, 5)


### 2.1. Generate Complete Training Data with All 9 Product Categories

**Note**: Instead of using the original 5-sample dataset, we generate synthetic data with all 9 product categories:
- pl (Personal Loan)
- hy (Hypothec/Mortgage)  
- ins (Insurance)
- cc (Credit Card)
- loan (General Loan)
- fund (Investment Fund)
- fx (Foreign Exchange)
- bond (Bond)
- deposit (Deposit)

In [5]:
# Set random seed for reproducibility
np.random.seed(42)

# Define product categories (9 different categories)
PRODUCT_CATEGORIES = ['pl', 'hy', 'ins', 'cc', 'loan', 'fund', 'fx', 'bond', 'deposit']

print("="*80)
print("PRODUCT CATEGORIES")
print("="*80)
print(f"Total categories: {len(PRODUCT_CATEGORIES)}")
for i, product in enumerate(PRODUCT_CATEGORIES):
    print(f"  {i}: {product}")

PRODUCT CATEGORIES
Total categories: 9
  0: pl
  1: hy
  2: ins
  3: cc
  4: loan
  5: fund
  6: fx
  7: bond
  8: deposit


In [6]:
def generate_share_feature(customer_segment):
    """
    Generate share_feature_1 based on customer segment
    
    Segments:
    - Low value: 50-200
    - Medium value: 200-500
    - High value: 500-1000
    - VIP: 1000-2000
    """
    if customer_segment == 'low':
        return np.random.randint(50, 200)
    elif customer_segment == 'medium':
        return np.random.randint(200, 500)
    elif customer_segment == 'high':
        return np.random.randint(500, 1000)
    else:  # VIP
        return np.random.randint(1000, 2000)

def generate_product_feature(product_id, customer_segment):
    """
    Generate product_feature_2 based on product type and customer segment
    """
    # Base ranges by product type
    product_ranges = {
        'pl': (20, 80),      # Personal loan
        'hy': (10, 70),      # Hypothec/mortgage
        'ins': (15, 60),     # Insurance
        'cc': (30, 90),      # Credit card
        'loan': (20, 75),    # General loan
        'fund': (10, 50),    # Investment fund
        'fx': (5, 40),       # Foreign exchange
        'bond': (5, 35),     # Bond
        'deposit': (40, 95)  # Deposit
    }
    
    min_val, max_val = product_ranges.get(product_id, (10, 60))
    
    # Adjust based on customer segment
    if customer_segment in ['high', 'vip']:
        min_val = int(min_val * 1.2)
        max_val = min(100, int(max_val * 1.3))
    
    return np.random.randint(min_val, max_val + 1)

def assign_relevance_score(share_feature, product_feature, product_id):
    """
    Assign relevance score based on feature patterns
    
    Logic:
    - Score 3 (successfully applied): High engagement + suitable product features
    - Score 2 (in process): Medium engagement + moderate product features
    - Score 1 (browsing only): Low engagement or unsuitable product features
    """
    # Calculate combined score
    combined_score = (share_feature / 2000) * 0.6 + (product_feature / 100) * 0.4
    
    # Add some randomness for realistic distribution
    noise = np.random.uniform(-0.15, 0.15)
    combined_score += noise
    
    # High-value products (pl, hy, loan) have different thresholds
    high_value_products = ['pl', 'hy', 'loan']
    
    if product_id in high_value_products:
        # More stringent criteria for high-value products
        if combined_score > 0.65:
            return 3
        elif combined_score > 0.40:
            return 2
        else:
            return 1
    else:
        # Standard criteria for other products
        if combined_score > 0.60:
            return 3
        elif combined_score > 0.35:
            return 2
        else:
            return 1

print("✅ Data generation functions defined")

✅ Data generation functions defined


In [7]:
# Generate training data with all 9 product categories
print("="*80)
print("GENERATING TRAINING DATA WITH ALL 9 CATEGORIES")
print("="*80)

num_customers = 20  # Small dataset but complete categories
products_per_customer_range = (5, 8)

# Define customer segments distribution
segments = ['low', 'medium', 'high', 'vip']
segment_weights = [0.3, 0.4, 0.2, 0.1]  # 30% low, 40% medium, 20% high, 10% VIP

data = []
customer_ids = [f'c{str(i+1).zfill(3)}' for i in range(num_customers)]

for cust_id in customer_ids:
    # Assign customer segment
    segment = np.random.choice(segments, p=segment_weights)
    share_feature = generate_share_feature(segment)
    
    # Determine number of products for this customer
    num_products = np.random.randint(products_per_customer_range[0], 
                                    products_per_customer_range[1] + 1)
    
    # Randomly select products (without replacement)
    selected_products = np.random.choice(PRODUCT_CATEGORIES, 
                                        size=num_products, 
                                        replace=False)
    
    for product_id in selected_products:
        product_feature = generate_product_feature(product_id, segment)
        relevance_score = assign_relevance_score(share_feature, product_feature, product_id)
        
        data.append({
            'cust_id': cust_id,
            'product_id': product_id,
            'share_feature_1': share_feature,
            'product_feature_2': product_feature,
            'relevance_score': relevance_score
        })

# Replace df with generated data
df = pd.DataFrame(data)

print(f"\n✅ Training data generated:")
print(f"   Total samples: {len(df)}")
print(f"   Unique customers: {df['cust_id'].nunique()}")
print(f"   Product categories: {df['product_id'].nunique()}")
print(f"\n📊 Product distribution:")
print(df['product_id'].value_counts().sort_index())
print(f"\n📊 Relevance score distribution:")
print(df['relevance_score'].value_counts().sort_index())
print(f"\nFirst few rows:")
display(df.head(10))

GENERATING TRAINING DATA WITH ALL 9 CATEGORIES

✅ Training data generated:
   Total samples: 134
   Unique customers: 20
   Product categories: 9

📊 Product distribution:
product_id
bond       17
cc         14
deposit    12
fund       15
fx         14
hy         16
ins        15
loan       14
pl         17
Name: count, dtype: int64

📊 Relevance score distribution:
relevance_score
1    93
2    27
3    14
Name: count, dtype: int64

First few rows:


,cust_id,product_id,share_feature_1,product_feature_2,relevance_score
0,c001,fund,470,33,1
1,c001,pl,470,59,2
2,c001,cc,470,51,1
3,c001,fx,470,28,2
4,c001,ins,470,52,1
5,c001,hy,470,69,1
6,c001,deposit,470,51,2
7,c002,hy,138,16,1
8,c002,fund,138,48,1
9,c002,loan,138,44,1


## 3. Feature Engineering

In [8]:
# Encode product_id (Label Encoding)
product_encoder = LabelEncoder()
df['product_id_encoded'] = product_encoder.fit_transform(df['product_id'])

print("✅ Product ID Encoding:")
for product, code in zip(product_encoder.classes_, range(len(product_encoder.classes_))):
    print(f"   {product} -> {code}")

feature_columns = ['product_id_encoded', 'share_feature_1', 'product_feature_2']
print(f"\n✅ Feature columns: {feature_columns}")
display(df[['cust_id', 'product_id', 'product_id_encoded', 'share_feature_1', 'product_feature_2', 'relevance_score']].head())

✅ Product ID Encoding:
   bond -> 0
   cc -> 1
   deposit -> 2
   fund -> 3
   fx -> 4
   hy -> 5
   ins -> 6
   loan -> 7
   pl -> 8

✅ Feature columns: ['product_id_encoded', 'share_feature_1', 'product_feature_2']


,cust_id,product_id,product_id_encoded,share_feature_1,product_feature_2,relevance_score
0,c001,fund,3,470,33,1
1,c001,pl,8,470,59,2
2,c001,cc,1,470,51,1
3,c001,fx,4,470,28,2
4,c001,ins,6,470,52,1


In [9]:
# Prepare training data
df = df.sort_values('cust_id').reset_index(drop=True)
X = df[feature_columns].values
y = df['relevance_score'].values
group_sizes = df.groupby('cust_id').size().values

print(f"✅ Training data prepared:")
print(f"   Total samples: {len(X)}")
print(f"   Number of groups: {len(group_sizes)}")
print(f"   Group sizes: {group_sizes}")

✅ Training data prepared:
   Total samples: 134
   Number of groups: 20
   Group sizes: [7 5 7 5 5 5 7 8 6 7 6 8 8 8 5 7 7 8 8 7]


## 4. Train LightGBM Model

In [10]:
# Train LightGBM
print("="*80)
print("TRAINING LIGHTGBM LAMBDAMART MODEL")
print("="*80)

lgb_params = {
    'objective': 'lambdarank',
    'metric': 'ndcg',
    'boosting_type': 'gbdt',
    'num_iterations': 100,  # Increased from 10 to 100
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': 6,  # Added depth constraint
    'min_data_in_leaf': 5,  # Minimum samples per leaf
    'feature_fraction': 0.8,  # Feature sampling
    'bagging_fraction': 0.8,  # Data sampling
    'bagging_freq': 5,  # Bagging frequency
    'lambda_l1': 0.1,  # L1 regularization
    'lambda_l2': 0.1,  # L2 regularization
    'verbose': -1  # Suppress warnings
}

print(f"\n📋 Parameters:")
for key, value in lgb_params.items():
    print(f"   {key}: {value}")

train_data = lgb.Dataset(X, label=y, group=group_sizes, feature_name=feature_columns)

print(f"\n🚀 Training with {lgb_params['num_iterations']} iterations...")

lgb_model = lgb.train(
    lgb_params,
    train_data,
    num_boost_round=lgb_params['num_iterations'],
    valid_sets=[train_data],
    valid_names=['train']
)

print("\n✅ LightGBM model trained")

# Get predictions
lgb_predictions = lgb_model.predict(X)
df['lgb_predicted_score'] = lgb_predictions

print("\n📊 Sample predictions:")
display(df[['cust_id', 'product_id', 'relevance_score', 'lgb_predicted_score']].head(10))

TRAINING LIGHTGBM LAMBDAMART MODEL

📋 Parameters:
   objective: lambdarank
   metric: ndcg
   boosting_type: gbdt
   num_iterations: 100
   learning_rate: 0.05
   num_leaves: 31
   max_depth: 6
   min_data_in_leaf: 5
   feature_fraction: 0.8
   bagging_fraction: 0.8
   bagging_freq: 5
   lambda_l1: 0.1
   lambda_l2: 0.1
   verbose: -1

🚀 Training with 100 iterations...

✅ LightGBM model trained

📊 Sample predictions:


,cust_id,product_id,relevance_score,lgb_predicted_score
0,c001,fund,1,-0.528308
1,c001,pl,2,1.082887
2,c001,cc,1,-0.096058
3,c001,fx,2,-0.010205
4,c001,ins,1,-0.994972
5,c001,hy,1,-0.451128
6,c001,deposit,2,0.220493
7,c002,hy,1,-2.394757
8,c002,fund,1,0.173766
9,c002,loan,1,-2.011387


In [11]:
# Save LightGBM model in multiple formats
timestamp = datetime.now(TZ_UTC8).strftime("%Y%m%d_%H%M%S")

# Save as pickle (for general use)
lgb_pkl_filename = f'ltr_lightgbm_model_{timestamp}.pkl'
with open(lgb_pkl_filename, 'wb') as f:
    pickle.dump(lgb_model, f)
print(f"✅ LightGBM model saved as pickle: {lgb_pkl_filename}")

# Save as JSON (for conversion to XGBoost)
lgb_json_filename = f'ltr_lightgbm_model_{timestamp}.json'
model_json_dump = lgb_model.dump_model()
with open(lgb_json_filename, 'w', encoding='utf-8') as f:
    json.dump(model_json_dump, f, indent=2, ensure_ascii=False)
print(f"✅ LightGBM model saved as JSON: {lgb_json_filename}")
print(f"\n⭐️ JSON file will be used for conversion to XGBoost!")

✅ LightGBM model saved as pickle: ltr_lightgbm_model_20260317_223914.pkl
✅ LightGBM model saved as JSON: ltr_lightgbm_model_20260317_223914.json

⭐️ JSON file will be used for conversion to XGBoost!


In [12]:
# List all local files with formatted output
list_files_formatted()

📁 LOCAL FILES

File Name                                                                 Size             Modified
--------------------------------------------------------------------------------------------------------------
📋 ltr_lightgbm_model_20260317_223914.json                            431.1 KB  2026-03-17 14:39:15
📦 ltr_lightgbm_model_20260317_223914.pkl                              90.9 KB  2026-03-17 14:39:14

✅ Total: 2 files


## 5. Convert to XGBoost

Convert the trained LightGBM model to XGBoost format using JSON-based tree structure conversion.

This process:
- Loads the LightGBM JSON dump
- Converts tree structures to XGBoost format
- Preserves model parameters including learning rate

In [13]:
# Conversion functions / 轉換函數
# 這些函數負責將 LightGBM 模型轉換為 XGBoost 格式
# These functions handle the conversion from LightGBM model to XGBoost format
from collections import deque

def load_lgb_model(path):
    """
    從 JSON 文件載入 LightGBM 模型
    Load LightGBM model from JSON file
    
    Args:
        path (str): LightGBM 模型 JSON 文件的路徑
                   Path to LightGBM model JSON file
    
    Returns:
        dict: LightGBM 模型的字典表示
             LightGBM model dump as dictionary
    
    Note:
        - 只支持 JSON 格式 (Only supports JSON format)
        - JSON 格式包含完整的樹結構信息 (JSON format contains complete tree structure information)
    """
    with open(path, "r") as f:
        return json.load(f)

def adjust_threshold(threshold):
    """
    調整閾值以適應 XGBoost 的分割邏輯
    Adjust threshold to match XGBoost's split logic
    
    LightGBM 使用 <= 進行分割，而 XGBoost 使用 <
    LightGBM uses <= for splits, while XGBoost uses <
    
    Args:
        threshold (float): LightGBM 的原始閾值
                          Original threshold from LightGBM
    
    Returns:
        float: 調整後的閾值，適用於 XGBoost
              Adjusted threshold for XGBoost
    
    Technical Details:
        - 使用 np.nextafter 獲取下一個可表示的浮點數
          Uses np.nextafter to get the next representable float
        - 這確保了 XGBoost 的 < 比較等同於 LightGBM 的 <= 比較
          This ensures XGBoost's < comparison is equivalent to LightGBM's <=
    """
    t = np.float32(threshold)
    return float(np.nextafter(t, np.float32(np.inf)))

def flatten_tree(root):
    """
    使用廣度優先搜索（BFS）將樹結構扁平化
    Flatten tree structure using Breadth-First Search (BFS)
    
    Args:
        root (dict): LightGBM 樹的根節點
                    Root node of LightGBM tree
    
    Returns:
        list: 扁平化的節點列表，每個元素包含節點和其父節點索引
             Flattened list of nodes, each containing node and parent index
    
    Algorithm:
        1. 使用隊列進行 BFS 遍歷 (Use queue for BFS traversal)
        2. 為每個節點分配連續的索引 (Assign sequential index to each node)
        3. 記錄父子關係 (Record parent-child relationships)
        4. 保持樹的層次結構 (Maintain tree hierarchy)
    
    Example:
        Input tree:     Output list:
            A           [A(parent=-1), B(parent=0), C(parent=0)]
           / \
          B   C
    """
    nodes = []
    queue = deque([(root, -1)])  # (節點, 父節點索引) / (node, parent index)
    
    while queue:
        node, parent = queue.popleft()
        node_id = len(nodes)  # 當前節點的索引 / Current node's index
        
        # 存儲節點及其父節點信息
        # Store node and its parent information
        nodes.append({
            "node": node,
            "parent": parent
        })
        
        # 如果是內部節點（有分割），將子節點加入隊列
        # If internal node (has split), add children to queue
        if "split_index" in node:
            queue.append((node["left_child"], node_id))
            queue.append((node["right_child"], node_id))
    
    return nodes

def build_arrays(nodes):
    """
    從扁平化的節點列表構建 XGBoost 所需的數組結構
    Build XGBoost array structures from flattened node list
    
    Args:
        nodes (list): 扁平化的節點列表（來自 flatten_tree）
                     Flattened node list (from flatten_tree)
    
    Returns:
        tuple: 包含 9 個數組的元組，對應 XGBoost 的內部表示
              Tuple of 9 arrays corresponding to XGBoost's internal representation:
              - left: 左子節點索引 (left child indices)
              - right: 右子節點索引 (right child indices)
              - parent: 父節點索引 (parent indices)
              - split_index: 分割特徵索引 (split feature indices)
              - split_cond: 分割條件/葉子值 (split conditions/leaf values)
              - default_left: 缺失值處理方向 (missing value direction)
              - base_weight: 基礎權重 (base weights)
              - sum_hessian: Hessian 總和 (sum of hessians)
              - loss_changes: 損失變化 (loss changes)
    
    XGBoost Internal Structure:
        - 內部節點：存儲分割信息 (Internal nodes: store split info)
        - 葉子節點：存儲預測值 (Leaf nodes: store prediction values)
        - 所有數組長度相同，等於節點總數 (All arrays same length = total nodes)
    """
    n = len(nodes)
    
    # 初始化所有數組 / Initialize all arrays
    left = [-1] * n          # -1 表示葉子節點 / -1 indicates leaf node
    right = [-1] * n         # -1 表示葉子節點 / -1 indicates leaf node
    parent = [-1] * n        # -1 表示根節點 / -1 indicates root node
    split_index = [0] * n    # 分割特徵的索引 / Index of split feature
    split_cond = [0.0] * n   # 分割閾值或葉子值 / Split threshold or leaf value
    default_left = [1] * n   # 1=左, 0=右 / 1=left, 0=right
    base_weight = [0.0] * n  # 節點權重 / Node weights
    sum_hessian = [1.0] * n  # Hessian 總和 / Sum of hessians
    loss_changes = [0.0] * n # 分割帶來的損失減少 / Loss reduction from split
    
    # 遍歷所有節點並填充數組
    # Iterate through all nodes and populate arrays
    for i, entry in enumerate(nodes):
        node = entry["node"]
        parent[i] = entry["parent"]
        
        # 檢查是否為內部節點（有分割）
        # Check if internal node (has split)
        if "split_index" in node:
            # === 內部節點處理 / Internal Node Processing ===
            
            # 1. 設置分割特徵和閾值
            #    Set split feature and threshold
            split_index[i] = node["split_feature"]
            split_cond[i] = adjust_threshold(node["threshold"])
            
            # 2. 設置缺失值處理方向
            #    Set missing value direction
            default_left[i] = 1 if node["default_left"] else 0
            
            # 3. 找到並設置左右子節點
            #    Find and set left/right children
            for j, child in enumerate(nodes):
                if child["parent"] == i:
                    if left[i] == -1:
                        left[i] = j  # 第一個子節點是左子節點 / First child is left
                    else:
                        right[i] = j  # 第二個子節點是右子節點 / Second child is right
            
            # 4. 設置分割增益和樣本數
            #    Set split gain and sample count
            loss_changes[i] = float(node.get("split_gain", 0.0))
            sum_hessian[i] = float(node.get("internal_count", 1))
        else:
            # === 葉子節點處理 / Leaf Node Processing ===
            
            # 關鍵：XGBoost 從 split_cond 讀取葉子值！
            # CRITICAL: XGBoost reads leaf values from split_cond!
            leaf_value = float(node["leaf_value"])
            base_weight[i] = leaf_value      # 存儲在 base_weight / Store in base_weight
            split_cond[i] = leaf_value       # 也必須存儲在 split_cond！/ Must also store in split_cond!
            sum_hessian[i] = float(node.get("leaf_count", 1))
    
    # 根節點沒有父節點 / Root node has no parent
    parent[0] = -1
    
    return (
        left,
        right,
        parent,
        split_index,
        split_cond,
        default_left,
        base_weight,
        sum_hessian,
        loss_changes,
    )

def convert_tree(tree_json, tree_id, num_feature):
    """
    將單個 LightGBM 樹轉換為 XGBoost 格式
    Convert a single LightGBM tree to XGBoost format
    
    Args:
        tree_json (dict): LightGBM 樹的 JSON 表示
                         JSON representation of LightGBM tree
        tree_id (int): 樹的 ID（在模型中的索引）
                      Tree ID (index in model)
        num_feature (int): 特徵總數
                          Total number of features
    
    Returns:
        dict: XGBoost 格式的樹結構
             Tree structure in XGBoost format
    
    Conversion Steps:
        1. 扁平化樹結構 (Flatten tree structure)
        2. 構建數組表示 (Build array representation)
        3. 創建 XGBoost 樹字典 (Create XGBoost tree dictionary)
        4. 設置樹參數 (Set tree parameters)
    """
    # 步驟 1: 扁平化樹結構
    # Step 1: Flatten tree structure
    nodes = flatten_tree(tree_json["tree_structure"])
    
    # 步驟 2: 構建數組表示
    # Step 2: Build array representation
    (
        left,
        right,
        parent,
        split_index,
        split_cond,
        default_left,
        base_weight,
        sum_hessian,
        loss_changes,
    ) = build_arrays(nodes)
    
    n = len(nodes)  # 節點總數 / Total number of nodes
    
    # 步驟 3 & 4: 創建 XGBoost 樹字典
    # Step 3 & 4: Create XGBoost tree dictionary
    return {
        "base_weights": base_weight,           # 節點權重 / Node weights
        "categories": [],                      # 類別特徵（未使用）/ Categorical features (unused)
        "categories_nodes": [],                # 類別節點（未使用）/ Category nodes (unused)
        "categories_segments": [],             # 類別段（未使用）/ Category segments (unused)
        "categories_sizes": [],                # 類別大小（未使用）/ Category sizes (unused)
        "default_left": default_left,          # 缺失值方向 / Missing value direction
        "id": tree_id,                         # 樹 ID / Tree ID
        "left_children": left,                 # 左子節點 / Left children
        "loss_changes": loss_changes,          # 損失變化 / Loss changes
        "parents": parent,                     # 父節點 / Parents
        "right_children": right,               # 右子節點 / Right children
        "split_conditions": split_cond,        # 分割條件 / Split conditions
        "split_indices": split_index,          # 分割特徵 / Split features
        "split_type": [0] * n,                 # 分割類型（0=數值）/ Split type (0=numerical)
        "sum_hessian": sum_hessian,           # Hessian 總和 / Sum of hessians
        "tree_param": {                        # 樹參數 / Tree parameters
            "num_deleted": "0",                # 刪除的節點數 / Number of deleted nodes
            "num_feature": str(num_feature),   # 特徵數 / Number of features
            "num_nodes": str(n),               # 節點數 / Number of nodes
            "size_leaf_vector": "0",           # 葉子向量大小 / Leaf vector size
        },
    }

def build_xgb_json(lgb_dump):
    """
    從 LightGBM dump 構建完整的 XGBoost JSON
    Build complete XGBoost JSON from LightGBM dump
    
    Args:
        lgb_dump (dict): LightGBM 模型的完整 dump
                        Complete dump of LightGBM model
    
    Returns:
        dict: XGBoost 格式的完整模型 JSON
             Complete model JSON in XGBoost format
    
    Structure:
        - learner: 學習器配置 (Learner configuration)
          - attributes: 屬性 (Attributes)
          - feature_names: 特徵名稱 (Feature names)
          - feature_types: 特徵類型 (Feature types)
          - gradient_booster: 梯度提升器 (Gradient booster)
            - model: 模型結構 (Model structure)
              - trees: 所有樹 (All trees)
          - learner_model_param: 學習器參數 (Learner parameters)
          - objective: 目標函數 (Objective function)
    """
    # 獲取特徵數量 / Get number of features
    num_feature = lgb_dump["max_feature_idx"] + 1
    
    # 轉換所有樹 / Convert all trees
    trees = []
    for i, tree in enumerate(lgb_dump["tree_info"]):
        trees.append(convert_tree(tree, i, num_feature))
    
    num_trees = len(trees)
    
    # 構建完整的 XGBoost JSON 結構
    # Build complete XGBoost JSON structure
    return {
        "learner": {
            "attributes": {},  # 額外屬性（可選）/ Additional attributes (optional)
            "feature_names": lgb_dump["feature_names"],  # 特徵名稱 / Feature names
            "feature_types": ["float"] * num_feature,    # 所有特徵都是浮點型 / All features are float
            "gradient_booster": {
                "name": "gbtree",  # 使用樹提升 / Use tree boosting
                "model": {
                    "gbtree_model_param": {
                        "num_parallel_tree": "1",      # 每次迭代一棵樹 / One tree per iteration
                        "num_trees": str(num_trees),   # 樹的總數 / Total number of trees
                    },
                    "iteration_indptr": list(range(num_trees + 1)),  # 迭代指針 / Iteration pointers
                    "tree_info": [0] * num_trees,    # 樹信息（全部為 0）/ Tree info (all 0)
                    "trees": trees,                   # 所有轉換後的樹 / All converted trees
                },
            },
            "learner_model_param": {
                "base_score": "0",              # 基礎分數 / Base score
                "boost_from_average": "0",      # 不從平均值開始 / Don't boost from average
                "num_class": "0",               # 類別數（0 表示回歸）/ Number of classes (0 for regression)
                "num_feature": str(num_feature), # 特徵數 / Number of features
                "num_target": "1",              # 目標數 / Number of targets
            },
            "objective": {"name": "rank:ndcg"},  # 排序目標：NDCG / Ranking objective: NDCG
        },
        "version": [2, 0, 3],  # XGBoost 版本 / XGBoost version
    }

def build_xgb_booster(model_json, learning_rate=1.0, temp_file=None):
    """
    從 JSON 構建 XGBoost Booster 對象
    Build XGBoost Booster object from JSON
    
    Args:
        model_json (dict): XGBoost 格式的模型 JSON
                          Model JSON in XGBoost format
        learning_rate (float): 學習率（默認 1.0）
                              Learning rate (default 1.0)
        temp_file (str): 臨時文件路徑（可選）
                        Temporary file path (optional)
    
    Returns:
        xgb.Booster: XGBoost Booster 對象
                    XGBoost Booster object
    
    Process:
        1. 將 JSON 寫入臨時文件 (Write JSON to temporary file)
        2. 創建 Booster 對象 (Create Booster object)
        3. 從文件載入模型 (Load model from file)
        4. 設置學習率 (Set learning rate)
    
    Note:
        - 學習率必須與原始 LightGBM 模型一致
          Learning rate must match original LightGBM model
        - 臨時文件用於 XGBoost 的內部載入機制
          Temporary file is used for XGBoost's internal loading mechanism
    """
    # 設置臨時文件路徑 / Set temporary file path
    if temp_file is None:
        temp_file = "tmp_xgb_model.json"
    
    # 步驟 1: 將 JSON 寫入文件
    # Step 1: Write JSON to file
    with open(temp_file, "w") as f:
        json.dump(model_json, f)
    
    # 步驟 2 & 3: 創建並載入 Booster
    # Step 2 & 3: Create and load Booster
    booster = xgb.Booster()
    booster.load_model(temp_file)
    
    # 步驟 4: 設置學習率（如果不是默認值）
    # Step 4: Set learning rate (if not default)
    if learning_rate != 1.0:
        booster.set_param('learning_rate', str(learning_rate))
    
    return booster

print("✅ Conversion functions defined / 轉換函數已定義")

✅ Conversion functions defined / 轉換函數已定義


In [14]:
# Convert LightGBM to XGBoost
print("="*80)
print("CONVERTING LIGHTGBM TO XGBOOST")
print("="*80)

conversion_successful = False

try:
    # Load the JSON dump directly
    print("Loading LightGBM JSON dump...")
    with open(lgb_json_filename, 'r', encoding='utf-8') as f:
        dump = json.load(f)
    
    # Convert trees to XGBoost format
    print("Converting trees...")
    model_json = build_xgb_json(dump)
    
    # Build XGBoost booster
    print("Building XGBoost booster...")
    temp_xgb_json = f'tmp_xgb_model_{timestamp}.json'
    xgb_model = build_xgb_booster(model_json, learning_rate=lgb_params['learning_rate'], temp_file=temp_xgb_json)
    
    print(f"\n✅ Conversion complete!")
    print(f"   Number of trees: {xgb_model.num_boosted_rounds()}")
    print(f"\n🎉 SUCCESS: Model converted successfully!")
    conversion_successful = True
    
except Exception as e:
    print(f"\n❌ CONVERSION FAILED")
    print(f"   Error: {e}")
    print(f"   Error type: {type(e).__name__}")
    import traceback
    traceback.print_exc()

CONVERTING LIGHTGBM TO XGBOOST
Loading LightGBM JSON dump...
Converting trees...
Building XGBoost booster...

✅ Conversion complete!
   Number of trees: 100

🎉 SUCCESS: Model converted successfully!


## 6. Verify Conversion

In [15]:
# Verify predictions
print("="*80)
print("VERIFYING MODEL CONVERSION")
print("="*80)

dtest = xgb.DMatrix(X, feature_names=feature_columns)
xgb_predictions = xgb_model.predict(dtest)
df['xgb_predicted_score'] = xgb_predictions

pred_diff = np.abs(lgb_predictions - xgb_predictions)
print(f"\n📊 Prediction Differences:")
print(f"   Max: {np.max(pred_diff):.6f}")
print(f"   Mean: {np.mean(pred_diff):.6f}")
print(f"   Std: {np.std(pred_diff):.6f}")

if np.max(pred_diff) < 0.1:
    print(f"\n✅ Models are functionally equivalent")
else:
    print(f"\n⚠️ Models show differences (expected due to different implementations)")

print("\n📊 Comparison:")
display(df[['cust_id', 'product_id', 'relevance_score', 'lgb_predicted_score', 'xgb_predicted_score']].head())

VERIFYING MODEL CONVERSION

📊 Prediction Differences:
   Max: 0.000001
   Mean: 0.000000
   Std: 0.000000

✅ Models are functionally equivalent

📊 Comparison:


,cust_id,product_id,relevance_score,lgb_predicted_score,xgb_predicted_score
0,c001,fund,1,-0.528308,-0.528308
1,c001,pl,2,1.082887,1.082887
2,c001,cc,1,-0.096058,-0.096058
3,c001,fx,2,-0.010205,-0.010205
4,c001,ins,1,-0.994972,-0.994972


## 7. Save XGBoost Model

In [16]:
# Save XGBoost model
print("="*80)
print("SAVING XGBOOST MODEL")
print("="*80)

model_filename = f'ltr_converted_xgboost_ranker_{timestamp}.pkl'
with open(model_filename, 'wb') as f:
    pickle.dump(xgb_model, f)
print(f"\n✅ Model saved: {model_filename}")

# Save encoders
encoders = {'product_id': product_encoder}
encoders_filename = f'ltr_converted_xgboost_encoders_{timestamp}.pkl'
with open(encoders_filename, 'wb') as f:
    pickle.dump(encoders, f)
print(f"✅ Encoders saved: {encoders_filename}")

# Save metadata
metadata = {
    'feature_columns': feature_columns,
    'product_encoding': dict(zip(product_encoder.classes_, range(len(product_encoder.classes_)))),
    'timestamp': timestamp,
    'xgboost_version': xgb_version,
    'lightgbm_version': lgb_version,
    'model_params': lgb_params,
    'deployment_type': 'xgboost_2.0',
    'python_version': '3.11'
}
metadata_filename = f'ltr_converted_xgboost_metadata_{timestamp}.pkl'
with open(metadata_filename, 'wb') as f:
    pickle.dump(metadata, f)
print(f"✅ Metadata saved: {metadata_filename}")

# Create tar.gz archive
archive_filename = f'ltr_converted_xgboost_ranker_{timestamp}.tar.gz'
with tarfile.open(archive_filename, 'w:gz') as tar:
    tar.add(model_filename, arcname=model_filename)
print(f"✅ Archive created: {archive_filename}")

print(f"\n📦 Files ready for deployment:")
print(f"   - {archive_filename}")
print(f"   - {metadata_filename}")

SAVING XGBOOST MODEL

✅ Model saved: ltr_converted_xgboost_ranker_20260317_223914.pkl
✅ Encoders saved: ltr_converted_xgboost_encoders_20260317_223914.pkl
✅ Metadata saved: ltr_converted_xgboost_metadata_20260317_223914.pkl
✅ Archive created: ltr_converted_xgboost_ranker_20260317_223914.tar.gz

📦 Files ready for deployment:
   - ltr_converted_xgboost_ranker_20260317_223914.tar.gz
   - ltr_converted_xgboost_metadata_20260317_223914.pkl


In [17]:
# List all local files with formatted outputlist_files_formatted()# List all local files with formatted output
list_files_formatted()

📁 LOCAL FILES

File Name                                                                 Size             Modified
--------------------------------------------------------------------------------------------------------------
🗜️ ltr_converted_xgboost_ranker_20260317_223914.tar.gz                 16.5 KB  2026-03-17 14:40:18
📦 ltr_converted_xgboost_encoders_20260317_223914.pkl                  742.0 B  2026-03-17 14:40:18
📦 ltr_converted_xgboost_metadata_20260317_223914.pkl                   1.1 KB  2026-03-17 14:40:18
📦 ltr_converted_xgboost_ranker_20260317_223914.pkl                   114.9 KB  2026-03-17 14:40:18
📋 tmp_xgb_model_20260317_223914.json                                 134.8 KB  2026-03-17 14:40:10
📋 ltr_lightgbm_model_20260317_223914.json                            431.1 KB  2026-03-17 14:39:15
📦 ltr_lightgbm_model_20260317_223914.pkl                              90.9 KB  2026-03-17 14:39:14

✅ Total: 7 files


## 8. Test Model Locally

In [18]:
# Test locally
print("🧪 Testing model locally...")

test_data = [[0, 100, 21], [2, 100, 32], [1, 100, 15]]
test_array = np.array(test_data)
dtest_local = xgb.DMatrix(test_array, feature_names=feature_columns)
local_predictions = xgb_model.predict(dtest_local)

print(f"\n✅ Local predictions:")
products = ['hy', 'pl', 'ins']
for p, s in zip(products, local_predictions):
    print(f"   {p}: {s:.6f}")

# Create scoring payload
scoring_payload = {"input_data": [{"values": test_data}]}
scoring_filename = f'ltr_scoring_input_{timestamp}.json'
with open(scoring_filename, 'w') as f:
    json.dump(scoring_payload, f, indent=2)
print(f"\n✅ Test payload saved: {scoring_filename}")

🧪 Testing model locally...

✅ Local predictions:
   hy: -1.916927
   pl: -0.498829
   ins: -1.458716

✅ Test payload saved: ltr_scoring_input_20260317_223914.json


## 9. Connect to watsonx.ai

### 9.1. Configuration

**IMPORTANT**: Update these values with your watsonx.ai credentials **ACCORDING TO YOUR ENVIRONMENT**.

#### 9.1.1. Environment: CP4D (on-prem)

In [ ]:
# ============================================================================
# CONFIGURATION - Update these values
# ============================================================================

username = 'PASTE YOUR USERNAME HERE'
url = 'PASTE THE PLATFORM URL HERE'

print("✅ Configuration loaded")

In [ ]:
# CP4D (on-prem)
from ibm_watsonx_ai import Credentials, APIClient
import getpass

credentials = Credentials(
    username=username,
    api_key=getpass.getpass("Enter your watsonx.ai api key and hit enter: "),
    url=url,
    instance_id="openshift",
    version="5.1"
)

client = APIClient(credentials)

print("✅ Connected to Watson Machine Learning")
print(f"   Version: {client.version}")

#### 9.1.2. Environment: IBM Cloud

In [19]:
# ============================================================================
# CONFIGURATION - Update these values
# ============================================================================

url = 'https://au-syd.ml.cloud.ibm.com'

print("✅ Configuration loaded")

✅ Configuration loaded


In [20]:
# IBM Cloud
from ibm_watsonx_ai import Credentials, APIClient
import getpass

credentials = Credentials(
    url=url,
    api_key=getpass.getpass("Enter your watsonx.ai api key and hit enter: "),
)

client = APIClient(credentials)

print("✅ Connected to Watson Machine Learning")
print(f"   Version: {client.version}")

Enter your watsonx.ai api key and hit enter:  ········


✅ Connected to Watson Machine Learning
   Version: 1.5.3


### 9.2. Set Default Target Project & Space

In [ ]:
# ============================================================================
# CONFIGURATION - Update these values
# ============================================================================

PROJECT_ID = "13139696-b110-479c-8cfb-6133779ce768"  # Replace with your project ID
SPACE_ID = "74fe39ae-d2e6-41e3-a51c-7ed1778e9d62"  # Replace with your deployment space ID

print("✅ Configuration loaded")
print(f"   PROJECT_ID: {PROJECT_ID}")
print(f"   SPACE_ID: {SPACE_ID}")

✅ Configuration loaded
   PROJECT_ID: 13139696-b110-479c-8cfb-6133779ce768
   SPACE_ID: 74fe39ae-d2e6-41e3-a51c-7ed1778e9d62


In [22]:
# Set default space
client.set.default_space(SPACE_ID)

print(f"✅ Default space set to: {SPACE_ID}")
space_details = client.spaces.get_details(SPACE_ID)
print(f"   Name: {space_details['entity']['name']}")

✅ Default space set to: 74fe39ae-d2e6-41e3-a51c-7ed1778e9d62
   Name: taishin_ai_gov_enablement_and_poc_demo_ds


## 10. Store Model in watsonx.ai

In [23]:
# Define model metadata
MODEL_NAME = model_filename

model_meta_props = {
    client.repository.ModelMetaNames.NAME: MODEL_NAME,
    client.repository.ModelMetaNames.TYPE: "xgboost_2.0",
    client.repository.ModelMetaNames.SOFTWARE_SPEC_UID: client.software_specifications.get_id_by_name("runtime-24.1-py3.11")
}

print("📋 Model metadata:")
print(f"   Model Name: {MODEL_NAME}")
print(f"   Type: xgboost_2.0")
print(f"   Software Spec: runtime-24.1-py3.11")

📋 Model metadata:
   Model Name: ltr_converted_xgboost_ranker_20260317_223914.pkl
   Type: xgboost_2.0
   Software Spec: runtime-24.1-py3.11


In [24]:
# Store model
print(f"\n🚀 Storing model in watsonx.ai...")
print(f"   Using archive: {archive_filename}")

published_model = client.repository.store_model(
    model=archive_filename,
    meta_props=model_meta_props
)

model_uid = client.repository.get_model_id(published_model)

print(f"\n✅ Model stored successfully!")
print(f"   Model UID: {model_uid}")


🚀 Storing model in watsonx.ai...
   Using archive: ltr_converted_xgboost_ranker_20260317_223914.tar.gz

✅ Model stored successfully!
   Model UID: b50970a7-ffe1-4ef8-8c64-9025d172c4b4


## 11. Deploy Model

In [25]:
# Define deployment metadata
DEPLOYMENT_NAME = model_filename + "_deployment"

deployment_meta_props = {
    client.deployments.ConfigurationMetaNames.NAME: DEPLOYMENT_NAME,
    client.deployments.ConfigurationMetaNames.ONLINE: {}
}

print("📋 Deployment metadata:")
print(f"   Deployment Name: {DEPLOYMENT_NAME}")
print(f"   Type: Online")
print(f"\n✅ Successfully defined deployment metadata")

📋 Deployment metadata:
   Deployment Name: ltr_converted_xgboost_ranker_20260317_223914.pkl_deployment
   Type: Online

✅ Successfully defined deployment metadata


In [26]:
# Create deployment
print(f"\n🚀 Creating deployment...")

deployment = client.deployments.create(
    artifact_uid=model_uid,
    meta_props=deployment_meta_props
)

deployment_uid = client.deployments.get_id(deployment)

print(f"\n✅ Deployment created successfully!")
print(f"   Deployment UID: {deployment_uid}")


🚀 Creating deployment...


######################################################################################

Synchronous deployment creation for id: 'b50970a7-ffe1-4ef8-8c64-9025d172c4b4' started

######################################################################################


initializing
Note: Scikit-learn 1.3/xgboost 2.0 framework with runtime-24.1-py3.11 for Watson Machine Learning is deprecated and will be removed in future. Use Scikit-learn 1.6/XGBoost 2.1 with runtime-25.1-py3.12 instead. For details, see https://dataplatform.cloud.ibm.com/docs/content/wsj/analyze-data/pm_service_supported_frameworks.html
.
ready


-----------------------------------------------------------------------------------------------
Successfully finished deployment creation, deployment_id='309ebd5c-36e6-4a84-9df2-faea0bbe4553'
-----------------------------------------------------------------------------------------------



✅ Deployment created successfully!
   Deployment UID: 309ebd5c-3

In [27]:
# Get deployment details
deployment_details = client.deployments.get_details(deployment_uid)

print("="*80)
print("DEPLOYMENT INFORMATION")
print("="*80)
print(f"\n📋 Deployment Details:")
print(f"   Name: {deployment_details['metadata']['name']}")
print(f"   Status: {deployment_details['entity']['status']['state']}")
print(f"   Deployment UID: {deployment_uid}")
print(f"   Model UID: {model_uid}")

scoring_endpoint = client.deployments.get_scoring_href(deployment_details)
print(f"\n🔗 Scoring Endpoint:")
print(f"   {scoring_endpoint}")

Note: Scikit-learn 1.3/xgboost 2.0 framework with runtime-24.1-py3.11 for Watson Machine Learning is deprecated and will be removed in future. Use Scikit-learn 1.6/XGBoost 2.1 with runtime-25.1-py3.12 instead. For details, see https://dataplatform.cloud.ibm.com/docs/content/wsj/analyze-data/pm_service_supported_frameworks.html
DEPLOYMENT INFORMATION

📋 Deployment Details:
   Name: ltr_converted_xgboost_ranker_20260317_223914.pkl_deployment
   Status: ready
   Deployment UID: 309ebd5c-36e6-4a84-9df2-faea0bbe4553
   Model UID: b50970a7-ffe1-4ef8-8c64-9025d172c4b4

🔗 Scoring Endpoint:
   https://au-syd.ml.cloud.ibm.com/ml/v4/deployments/309ebd5c-36e6-4a84-9df2-faea0bbe4553/predictions


## 12. Test Deployed Model

In [28]:
# Test deployment
print("🧪 Testing deployment...")
print(f"\nInput:")
print(scoring_payload)

predictions = client.deployments.score(deployment_uid, scoring_payload)

print(f"\n✅ Prediction successful!")
print(f"\nOutput:")
print(json.dumps(predictions, indent=2))

# Save test results
test_result_file = f'ltr_test_result_{timestamp}.json'
with open(test_result_file, 'w') as f:
    json.dump(predictions, f, indent=2)

print(f"\n✅ Test results saved: {test_result_file}")

🧪 Testing deployment...

Input:
{'input_data': [{'values': [[0, 100, 21], [2, 100, 32], [1, 100, 15]]}]}

✅ Prediction successful!

Output:
{
  "predictions": [
    {
      "fields": [
        "prediction",
        "probability"
      ],
      "values": [
        [
          -1.916926622390747,
          -1.916926622390747
        ],
        [
          -0.4988290071487427,
          -0.4988290071487427
        ],
        [
          -1.4587159156799316,
          -1.4587159156799316
        ]
      ]
    }
  ]
}

✅ Test results saved: ltr_test_result_20260317_223914.json


## 13. Summary

In [29]:
print("="*80)
print("✅ PIPELINE COMPLETED SUCCESSFULLY!")
print("="*80)

print(f"\n📊 Summary:")
print(f"   ✅ LightGBM model trained")
print(f"   ✅ Converted to XGBoost")
print(f"   ✅ Model verification completed")
print(f"   ✅ Deployed to watsonx.ai")
print(f"   ✅ Deployment tested successfully")

print(f"\n📦 Generated Files:")
print(f"   - Model: {archive_filename}")
print(f"   - Metadata: {metadata_filename}")
print(f"   - Test Input: {scoring_filename}")
print(f"   - Test Result: {test_result_file}")

print(f"\n📋 Deployment Info:")
print(f"   - Model UID: {model_uid}")
print(f"   - Deployment UID: {deployment_uid}")
print(f"   - Deployment Name: {DEPLOYMENT_NAME}")
print(f"   - Scoring Endpoint: {scoring_endpoint}")

✅ PIPELINE COMPLETED SUCCESSFULLY!

📊 Summary:
   ✅ LightGBM model trained
   ✅ Converted to XGBoost
   ✅ Model verification completed
   ✅ Deployed to watsonx.ai
   ✅ Deployment tested successfully

📦 Generated Files:
   - Model: ltr_converted_xgboost_ranker_20260317_223914.tar.gz
   - Metadata: ltr_converted_xgboost_metadata_20260317_223914.pkl
   - Test Input: ltr_scoring_input_20260317_223914.json
   - Test Result: ltr_test_result_20260317_223914.json

📋 Deployment Info:
   - Model UID: b50970a7-ffe1-4ef8-8c64-9025d172c4b4
   - Deployment UID: 309ebd5c-36e6-4a84-9df2-faea0bbe4553
   - Deployment Name: ltr_converted_xgboost_ranker_20260317_223914.pkl_deployment
   - Scoring Endpoint: https://au-syd.ml.cloud.ibm.com/ml/v4/deployments/309ebd5c-36e6-4a84-9df2-faea0bbe4553/predictions
